# Dunnhumby M1 후보상품 관계 진단
기존 seed-42 M1 체크포인트에서 Top-10 오추천과 누락 정답을 짝지어 N 후보관계와 V 가격적합성의 방향을 확인합니다. 재학습·최종 test·holdout은 수행하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'aa13ebd'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_candidate_relation_diagnostic import (
    configure_candidate_relation_diagnostic,
    preflight_summary,
    run_candidate_relation_diagnostic,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_candidate_relation_diagnostic('dunnhumby')
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
paths = run_candidate_relation_diagnostic(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['relation_summary_csv'])
focus = summary.group_type.isin(['overall', 'fixed_clv_segment', 'high_clv_composition'])
columns = [
    'signal', 'group_type', 'group', 'n_users', 'candidate_pair_count',
    'pair_balanced_win_rate', 'pair_strict_win_rate', 'pair_tie_rate',
    'mean_pair_score_difference',
]
print('1) 누락 정답이 M1 Top-10 오추천을 이기는 비율')
display(summary.loc[focus, columns])
print('판독 기준: 같은 관계가 Dunnhumby와 H&M 모두에서 전체·고CLV 승률 0.5 초과')
print('결과 파일:', paths)